# SDS 09 — SimRank, Direct Discovery, Spectral Partitioning & Overlapping Communities
Reusable reference notebook. Pure-Python cells are designed for intuition/tests; Spark sections use built-in PySpark only. Full pairwise SimRank and full eigensystems are inherently expensive, so scalable patterns explicitly bound state.

In [ ]:
import math, random
from collections import defaultdict, deque

## 1. Pure-Python SimRank reference

In [ ]:
def build_in_neighbors(nodes, edges):
    I={u:set() for u in nodes}
    for src,dst in edges:
        I.setdefault(src,set()); I.setdefault(dst,set()).add(src)
    return I

def simrank_local(nodes, edges, C=0.8, tol=1e-8, max_iter=100):
    nodes=list(nodes); I=build_in_neighbors(nodes,edges)
    S={(a,b):(1.0 if a==b else 0.0) for a in nodes for b in nodes}
    history=[]
    for it in range(max_iter):
        S2={}
        maxdiff=0.0
        for a in nodes:
            for b in nodes:
                if a==b:
                    val=1.0
                elif not I[a] or not I[b]:
                    val=0.0
                else:
                    total=sum(S[(x,y)] for x in I[a] for y in I[b])
                    val=C*total/(len(I[a])*len(I[b]))
                S2[(a,b)]=val
                maxdiff=max(maxdiff,abs(val-S[(a,b)]))
        S=S2; history.append(maxdiff)
        if maxdiff < tol: break
    return S,history

In [ ]:
nodes=list('ABCXYZ')
edges=[('A','X'),('B','X'),('A','Y'),('B','Y'),('A','Z'),('C','Z')]
S,hist=simrank_local(nodes,edges,C=0.8,tol=1e-12)
print('iterations:',len(hist),'last residual:',hist[-1])
print('s(X,Y)=',S[('X','Y')])
print('s(X,Z)=',S[('X','Z')])
assert abs(S[('X','Y')]-0.4)<1e-12
assert abs(S[('X','Z')]-0.2)<1e-12

## 2. Threshold SimRank into a similarity graph — this is an EXTRA clustering step

In [ ]:
def threshold_similarity_components(nodes,S,theta):
    adj={u:set() for u in nodes}
    for i,u in enumerate(nodes):
        for v in nodes[i+1:]:
            if S.get((u,v),S.get((v,u),0.0)) >= theta:
                adj[u].add(v); adj[v].add(u)
    seen=set(); comps=[]
    for s in nodes:
        if s in seen: continue
        q=deque([s]); seen.add(s); comp=[]
        while q:
            u=q.popleft(); comp.append(u)
            for v in adj[u]:
                if v not in seen:
                    seen.add(v); q.append(v)
        comps.append(sorted(comp))
    return comps

for theta in [0.15,0.25,0.35,0.45]:
    print(theta, threshold_similarity_components(nodes,S,theta))

## 3. Direct biclique evidence: common neighbors in a bipartite graph

In [ ]:
def k2t_candidates(edges, min_t=2):
    # edges are (left,right)
    right_to_left=defaultdict(set)
    for u,r in edges: right_to_left[r].add(u)
    shared=defaultdict(set)
    for r,lefts in right_to_left.items():
        ls=sorted(lefts)
        for i in range(len(ls)):
            for j in range(i+1,len(ls)):
                shared[(ls[i],ls[j])].add(r)
    return {pair:rs for pair,rs in shared.items() if len(rs)>=min_t}

bip=[('u1','r1'),('u1','r2'),('u1','r3'),('u2','r1'),('u2','r2'),('u2','r3'),('u3','r1')]
print(k2t_candidates(bip,2))

## 4. Clique percolation from a supplied list of k-cliques

In [ ]:
def clique_percolation(k_cliques,k):
    cliques=[set(c) for c in k_cliques]
    cadj={i:set() for i in range(len(cliques))}
    for i in range(len(cliques)):
        for j in range(i+1,len(cliques)):
            if len(cliques[i] & cliques[j]) >= k-1:
                cadj[i].add(j); cadj[j].add(i)
    seen=set(); communities=[]
    for s in range(len(cliques)):
        if s in seen: continue
        q=deque([s]); seen.add(s); ids=[]
        while q:
            i=q.popleft(); ids.append(i)
            for j in cadj[i]:
                if j not in seen:
                    seen.add(j); q.append(j)
        communities.append(set().union(*(cliques[i] for i in ids)))
    return communities

triangles=[{'A','B','C'},{'B','C','D'},{'C','E','F'},{'E','F','G'}]
comms=clique_percolation(triangles,3)
print(comms)
# C belongs to both communities, demonstrating overlap
assert sum('C' in c for c in comms)==2

## 5. AGM and BigCLAM probability helpers

In [ ]:
def agm_edge_probability(shared_community_ps):
    q=1.0
    for p in shared_community_ps: q*=1.0-p
    return 1.0-q

def bigclam_edge_probability(fu,fv):
    dot=sum(a*b for a,b in zip(fu,fv))
    return 1.0-math.exp(-dot)

print('AGM p:',agm_edge_probability([0.5,0.4]))
print('BigCLAM p:',bigclam_edge_probability([1.2,0.1],[0.8,0.4]))
assert abs(agm_edge_probability([0.5,0.4])-0.7)<1e-12

## 6. BigCLAM log-likelihood on a tiny graph (reference, not a full optimizer)

In [ ]:
def bigclam_loglikelihood(F, undirected_edges):
    nodes=sorted(F); E={tuple(sorted(e)) for e in undirected_edges}
    ll=0.0
    for i,u in enumerate(nodes):
        for v in nodes[i+1:]:
            dot=sum(a*b for a,b in zip(F[u],F[v]))
            if (u,v) in E:
                p=1.0-math.exp(-dot)
                ll += math.log(max(p,1e-300))
            else:
                ll -= dot  # log(exp(-dot))
    return ll

F={'A':[1.0,0.0],'B':[0.8,0.2],'C':[0.0,1.0]}
print('tiny LL:',bigclam_loglikelihood(F,[('A','B'),('B','C')]))

## 7. Spectral intuition helpers: Laplacian quadratic form and normalized cut

In [ ]:
def undirected_degree(nodes,edges):
    d={u:0 for u in nodes}
    for u,v in edges: d[u]+=1; d[v]+=1
    return d

def cut_size(S,edges):
    S=set(S); return sum((u in S)!=(v in S) for u,v in edges)

def normalized_cut(S,nodes,edges):
    S=set(S); T=set(nodes)-S; d=undirected_degree(nodes,edges)
    cut=cut_size(S,edges); volS=sum(d[u] for u in S); volT=sum(d[u] for u in T)
    return float('inf') if volS==0 or volT==0 else cut/volS + cut/volT

def laplacian_quadratic(x,edges):
    return sum((x[u]-x[v])**2 for u,v in edges)

N=list('ABCDEF')
E=[('A','B'),('A','C'),('B','C'),('C','D'),('D','E'),('D','F'),('E','F')]
print('Ncut ABC | DEF =',normalized_cut({'A','B','C'},N,E))
print('x^T L x for +/- indicator =',laplacian_quadratic({u:(-1 if u in 'ABC' else 1) for u in N},E))

---
# Spark-only reference patterns
These sections require PySpark. They intentionally keep large edge processing distributed.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window

## 8. Clean directed graph and choose a deliberately bounded core

In [ ]:
def load_clickstream_directed(path='pageviews.csv'):
    raw=(spark.read.option('sep','\t').option('header',False).csv(path)
         .toDF('src','dst','type','count'))
    return (raw.filter(F.col('type')=='link')
            .select('src','dst')
            .filter(F.col('src').isNotNull() & F.col('dst').isNotNull())
            .filter(F.col('src')!=F.col('dst'))
            .distinct().cache())

def top_total_degree_core(edges,K=500):
    outd=edges.groupBy('src').count().select(F.col('src').alias('node'),F.col('count').alias('outd'))
    ind=edges.groupBy('dst').count().select(F.col('dst').alias('node'),F.col('count').alias('ind'))
    top=(outd.join(ind,'node','outer').fillna(0)
         .withColumn('degree',F.col('outd')+F.col('ind'))
         .orderBy(F.desc('degree'),F.asc('node')).limit(K).select('node').cache())
    core=(edges.join(top.select(F.col('node').alias('src')),'src')
          .join(top.select(F.col('node').alias('dst')),'dst')
          .select('src','dst').distinct().cache())
    return top,core

## 9. Distributed SimRank on a SMALL core — explicit O(K^2) pair state

In [ ]:
def simrank_spark_core(core_edges, nodes_df, C=0.8, tol=1e-5, max_iter=20):
    # Only use when K^2 pair state is deliberately manageable.
    nodes=nodes_df.select(F.col('node').alias('a')).distinct().cache()
    nodes_b=nodes.select(F.col('a').alias('b'))
    pairs=nodes.crossJoin(nodes_b).cache()

    incoming=core_edges.select(F.col('dst').alias('node'),F.col('src').alias('pred')).cache()
    indeg=incoming.groupBy('node').count().withColumnRenamed('count','indeg').cache()

    scores=(pairs.withColumn('sim',F.when(F.col('a')==F.col('b'),F.lit(1.0)).otherwise(F.lit(0.0)))
            .cache())
    for it in range(max_iter):
        propagated=(scores.alias('s')
            .join(incoming.alias('ia'),F.col('s.a')==F.col('ia.pred'))
            .join(incoming.alias('ib'),F.col('s.b')==F.col('ib.pred'))
            .groupBy(F.col('ia.node').alias('a'),F.col('ib.node').alias('b'))
            .agg(F.sum('s.sim').alias('sum_sim')))
        da=indeg.select(F.col('node').alias('a'),F.col('indeg').alias('da'))
        db=indeg.select(F.col('node').alias('b'),F.col('indeg').alias('db'))
        new=(pairs.join(propagated,['a','b'],'left').join(da,'a','left').join(db,'b','left')
             .fillna(0.0,subset=['sum_sim'])
             .withColumn('sim',
                 F.when(F.col('a')==F.col('b'),F.lit(1.0))
                  .when(F.col('da').isNull() | F.col('db').isNull(),F.lit(0.0))
                  .otherwise(F.lit(C)*F.col('sum_sim')/(F.col('da')*F.col('db'))))
             .select('a','b','sim').cache())
        diff=(new.alias('n').join(scores.alias('o'),['a','b'])
              .agg(F.max(F.abs(F.col('n.sim')-F.col('o.sim'))).alias('diff')).first()['diff'] or 0.0)
        scores.unpersist(); scores=new
        if diff < tol: return scores,it+1,float(diff)
    return scores,max_iter,float(diff)

## 10. Threshold SimRank and compute components by iterative label propagation

In [ ]:
def threshold_sim_edges(scores,theta):
    return (scores.filter((F.col('a')<F.col('b')) & (F.col('sim')>=theta))
            .select(F.col('a').alias('u'),F.col('b').alias('v')).distinct())

def connected_components_labels(nodes_df, undirected_edges, max_iter=100):
    # Built-in Spark-only min-label propagation. Suitable as a reference; may require many rounds.
    nodes=nodes_df.select(F.col('node')).distinct()
    E=(undirected_edges.select(F.col('u').alias('src'),F.col('v').alias('dst'))
       .union(undirected_edges.select(F.col('v').alias('src'),F.col('u').alias('dst'))).distinct().cache())
    labels=nodes.withColumn('label',F.col('node')).cache()
    for _ in range(max_iter):
        msgs=E.join(labels.select(F.col('node').alias('src'),'label'),'src').select(F.col('dst').alias('node'),'label')
        best=msgs.groupBy('node').agg(F.min('label').alias('nbr_label'))
        new=(labels.join(best,'node','left')
             .withColumn('new_label',F.least(F.col('label'),F.coalesce(F.col('nbr_label'),F.col('label'))))
             .select('node',F.col('new_label').alias('label')).cache())
        changed=new.alias('n').join(labels.alias('o'),'node').filter(F.col('n.label')!=F.col('o.label')).limit(1).count()
        labels.unpersist(); labels=new
        if changed==0: break
    return labels

## 11. Spark K2,t biclique candidates

In [ ]:
def spark_k2t_candidates(bipartite_edges,min_t=2):
    E=bipartite_edges.select('left','right').distinct()
    pairs=(E.alias('a').join(E.alias('b'),F.col('a.right')==F.col('b.right'))
           .filter(F.col('a.left')<F.col('b.left'))
           .select(F.col('a.left').alias('u'),F.col('b.left').alias('v'),F.col('a.right').alias('r')))
    return (pairs.groupBy('u','v').agg(F.countDistinct('r').alias('t'))
            .filter(F.col('t')>=min_t))

## 12. Sparse normalized-adjacency matvec for spectral partitioning

In [ ]:
def symmetrize_canonical(U):
    # U canonical undirected (u,v) with u<v
    return (U.select(F.col('u').alias('src'),F.col('v').alias('dst'))
            .union(U.select(F.col('v').alias('src'),F.col('u').alias('dst'))).distinct())

def undirected_degree_df(U):
    return (U.select(F.col('u').alias('node')).union(U.select(F.col('v').alias('node')))
            .groupBy('node').count().withColumnRenamed('count','degree'))

def normalized_adjacency_matvec(U,x):
    E=symmetrize_canonical(U)
    d=undirected_degree_df(U)
    ds=d.select(F.col('node').alias('src'),F.col('degree').alias('ds'))
    dd=d.select(F.col('node').alias('dst'),F.col('degree').alias('dd'))
    xs=x.select(F.col('node').alias('src'),F.col('x').alias('xsrc'))
    return (E.join(ds,'src').join(dd,'dst').join(xs,'src')
            .withColumn('contrib',F.col('xsrc')/F.sqrt(F.col('ds')*F.col('dd')))
            .groupBy(F.col('dst').alias('node')).agg(F.sum('contrib').alias('x')))

## 13. Second eigenvector iteration with orthogonalization to q=sqrt(degree)

In [ ]:
def spectral_second_vector(U,seed=42,tol=1e-5,max_iter=100):
    d=undirected_degree_df(U).cache()
    # deterministic pseudo-random start from xxhash64; no Python hash()
    x=(d.withColumn('x',((F.pmod(F.xxhash64('node'),F.lit(2000001))/F.lit(1000000.0))-1.0))
       .select('node','x').cache())
    q=d.withColumn('q',F.sqrt('degree')).select('node','q').cache()
    qq=q.agg(F.sum(F.col('q')*F.col('q')).alias('s')).first()['s']
    for it in range(max_iter):
        # Apply lazy/shifted M=(I+S)/2 so power iteration targets the
        # second-largest algebraic eigenvector of S after deflating q.
        sx=normalized_adjacency_matvec(U,x)
        y=(sx.alias('s').join(x.alias('o'),'node')
           .select('node',(F.lit(0.5)*F.col('s.x')+F.lit(0.5)*F.col('o.x')).alias('x')))
        alpha=(y.join(q,'node').agg(F.sum(F.col('x')*F.col('q')).alias('s')).first()['s'] or 0.0)/qq
        y=(y.join(q,'node').withColumn('x',F.col('x')-F.lit(alpha)*F.col('q')).select('node','x'))
        norm=math.sqrt(y.agg(F.sum(F.col('x')*F.col('x')).alias('s')).first()['s'] or 1.0)
        y=y.withColumn('x',F.col('x')/F.lit(norm)).cache()
        diff=(y.alias('n').join(x.alias('o'),'node')
              .agg(F.sum(F.abs(F.col('n.x')-F.col('o.x'))).alias('d')).first()['d'] or 0.0)
        x.unpersist(); x=y
        if diff<tol: return x,it+1,float(diff)
    return x,max_iter,float(diff)

## 14. Simple sign partition and normalized-cut scoring in Spark

In [ ]:
def sign_partition(vec):
    return vec.withColumn('community',F.when(F.col('x')<0,F.lit(0)).otherwise(F.lit(1))).select('node','community')

def normalized_cut_spark(U,assignments):
    d=undirected_degree_df(U); vol=(d.join(assignments,'node').groupBy('community').agg(F.sum('degree').alias('vol')))
    au=assignments.select(F.col('node').alias('u'),F.col('community').alias('cu'))
    av=assignments.select(F.col('node').alias('v'),F.col('community').alias('cv'))
    cut=U.join(au,'u').join(av,'v').filter(F.col('cu')!=F.col('cv')).count()
    vols={r['community']:r['vol'] for r in vol.collect()}  # only 2 scalar groups
    if len(vols)!=2 or min(vols.values())==0: return float('inf')
    vals=list(vols.values())
    return cut/vals[0] + cut/vals[1]

## 15. Exam checklist
1. Say whether output is similarity, disjoint partition, or overlapping membership.
2. SimRank: state C, in-neighbor recurrence, tolerance/MAX_IT, and O(K^2) core size.
3. If thresholding SimRank, explicitly call it an added heuristic and justify theta.
4. Direct discovery: clique vs biclique; disclose combinatorial/self-join blow-up.
5. Spectral: Ncut, L=D-A, Fiedler/second normalized eigenvector; use sparse matvec.
6. Overlap: do not force one label per node; state k or affiliation threshold/model parameters.
7. Keep large preprocessing in Spark; collect only bounded state/scalars.